# Deep agent backends: State, Filesystem, Store

Each backend controls **where the agent's built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`) store their data.

| Backend | Where files live | Lifetime |
|---|---|---|
| `StateBackend` | LangGraph state (`files` key) | Per thread, ephemeral |
| `FilesystemBackend` | Real disk under `root_dir` | Permanent, on disk |
| `StoreBackend` | LangGraph `BaseStore` | Across threads / conversations |
| `CompositeBackend` | Routes by path prefix | Mix of the above |

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend, StoreBackend

load_dotenv()

model = ChatGroq(model="openai/gpt-oss-120b")


def ask(agent, text, thread_id="1"):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    print(result["messages"][-1].content)
    return result

## 1. StateBackend (default)

Files are kept in the agent's LangGraph state and checkpointed after every step. They persist across turns **within one thread** but vanish when the thread ends. A `checkpointer` is required for multi-turn memory.

In [ ]:
state_agent = create_deep_agent(
    model=model,
    backend=StateBackend(),
    checkpointer=MemorySaver(),
    system_prompt="You are a note taker. Use write_file / read_file to manage notes.",
)

ask(state_agent, "Write a file /notes.md containing 'Deep agents rock'.", thread_id="state-1")

In [ ]:
ask(state_agent, "Read /notes.md back to me.", thread_id="state-1")

In [ ]:
result = ask(state_agent, "List the files you have.", thread_id="state-1")
print("\nFiles held in state:", list(result.get("files", {}).keys()))

In [ ]:
# A different thread has no access to those files
ask(state_agent, "Read /notes.md.", thread_id="state-2")

## 2. FilesystemBackend

Files are real files on disk. `root_dir` is the sandbox root and `virtual_mode=True` means the agent sees `/hello.txt`, which maps to `<root_dir>/hello.txt`.

> Only use this in local / trusted environments. It gives the agent real disk access.

In [ ]:
workspace = Path("workspace")
workspace.mkdir(exist_ok=True)

fs_agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend(root_dir=workspace, virtual_mode=True),
    system_prompt="You are a note taker. Use write_file / read_file to manage notes.",
)

ask(fs_agent, "Write a file /hello.txt containing 'Hello from the filesystem backend'.")

In [ ]:
print("On disk:", [p.name for p in workspace.iterdir()])
print("Content:", (workspace / "hello.txt").read_text())

## 3. StoreBackend

Files live in a LangGraph `BaseStore`, scoped by a `namespace` you supply (per user, per assistant, etc). They persist **across threads and conversations**. `InMemoryStore` is used here; swap in a Postgres or Redis store for production.

In [ ]:
store = InMemoryStore()

store_agent = create_deep_agent(
    model=model,
    backend=StoreBackend(store=store, namespace=lambda runtime: ("memories",)),
    store=store,
    checkpointer=MemorySaver(),
    system_prompt="You are a note taker. Use write_file / read_file to manage notes.",
)

ask(store_agent, "Write a file /memory.md containing 'User likes cricket'.", thread_id="store-1")

In [ ]:
# A brand-new thread can still read it because the store is shared
ask(store_agent, "Read /memory.md and tell me what it says.", thread_id="store-2")

In [ ]:
print("Keys in store namespace ('memories',):", [item.key for item in store.search(("memories",))])

## 4. Bonus: CompositeBackend

Route paths to different backends by prefix. Here `/memories/...` goes to the `StoreBackend` (long-term) and everything else stays in `StateBackend` (scratch).

In [ ]:
composite_agent = create_deep_agent(
    model=model,
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(store=store, namespace=lambda rt: ("memories",))},
    ),
    store=store,
    checkpointer=MemorySaver(),
    system_prompt="You are a note taker. Scratch files go anywhere; long-term facts go under /memories/.",
)

ask(composite_agent, "Save '/scratch.txt' with 'temp' and '/memories/prefs.md' with 'prefers uv'.", thread_id="c-1")
print("\nStore now has:", [item.key for item in store.search(("memories",))])